# Exercícios

Utilizar a página https://keithgalli.github.io/web-scraping/webpage.html

## Carregar a pagina web


Carregando a página que será utilizada para resolver os exercícios.

In [1]:
from bs4 import BeautifulSoup as bs
import requests

r = requests.get('https://keithgalli.github.io/web-scraping/webpage.html')
webpage = bs(r.content)

print(webpage.head.title.prettify()) # Mostrando o titulo para ver se carregou corretamente.

<title>
 Keith Galli's Page
</title>



### 1 - Buscar todos os _links_ de rede social da página web.

Faça isso de três maneiras diferentes.

_Minha solução_

In [2]:
socials = list(map(lambda a: a['href'], webpage.select('b + a'))) # Escolhendo links (`a`) que sucedem `b` conforme 
                                                                    # a estrutura da página.
print(f'#1 {socials} \n')


socials = list(map(lambda li: li.a['href'], webpage.select('.social'))) # Escolhendo itens com a classe CSS `social`.
print(f'#2 {socials} \n')


a_tags = webpage.select('li > [href^="https"]') # Elementos que estão abaixo de `li` (_child_ de `li`) onde o atributo 
                                        # `href`, ou seja o link, começa (`^=`) com `"https"` conforme a 
                                        # documentação [4].

socials = list(map(lambda a: a['href'], a_tags)) # As `lambdas` foram utilizadas para percorrer os items utilizando 
                                                    # `map` ao invés de utilizar o laço `for`.
print(f'#3 {socials} \n')           

#1 ['https://www.instagram.com/keithgalli/', 'https://twitter.com/keithgalli', 'https://www.linkedin.com/in/keithgalli/', 'https://www.tiktok.com/@keithgalli'] 

#2 ['https://www.instagram.com/keithgalli/', 'https://twitter.com/keithgalli', 'https://www.linkedin.com/in/keithgalli/', 'https://www.tiktok.com/@keithgalli'] 

#3 ['https://www.instagram.com/keithgalli/', 'https://twitter.com/keithgalli', 'https://www.linkedin.com/in/keithgalli/', 'https://www.tiktok.com/@keithgalli'] 



_solução do vídeo_

In [3]:
links = webpage.select("ul.socials a") 
actual_links = [link['href'] for link in links] # Na solução do vídeo foi utilizado `list comprehensions` mais
                                                # interessante do que usar `lambda` por ser mais simples de visualizar.
actual_links

['https://www.instagram.com/keithgalli/',
 'https://twitter.com/keithgalli',
 'https://www.linkedin.com/in/keithgalli/',
 'https://www.tiktok.com/@keithgalli']

In [4]:
ulist = webpage.find("ul", attrs={"class": "socials"}) # utilizando o `find` que retorna um elemento
links = ulist.find_all("a") # Dentro desse elemento filtrado há os links.
actual_links = [link['href'] for link in links]
actual_links

['https://www.instagram.com/keithgalli/',
 'https://twitter.com/keithgalli',
 'https://www.linkedin.com/in/keithgalli/',
 'https://www.tiktok.com/@keithgalli']

In [5]:
links = webpage.select('li.social a')
actual_links = [link['href'] for link in links]
actual_links

['https://www.instagram.com/keithgalli/',
 'https://twitter.com/keithgalli',
 'https://www.linkedin.com/in/keithgalli/',
 'https://www.tiktok.com/@keithgalli']

### 2 - Scrape the HTML table into a Pandas `Dataframe`

_minha solução_

In [6]:
import pandas as pd

table_df = pd.read_html('https://keithgalli.github.io/web-scraping/webpage.html')[0]

table_df.head()

# Feito conforme aprendido nas aulas de Pandas.

,S,Team,League,GP,G,A,TP,PIM,+/-,Unnamed: 9,POST,GP.1,G.1,A.1,TP.1,PIM.1,+/-.1
0,2014-15,MIT (Mass. Inst. of Tech.),ACHA II,17.0,3.0,9.0,12.0,20.0,NaN,|,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2015-16,MIT (Mass. Inst. of Tech.),ACHA II,9.0,1.0,1.0,2.0,2.0,NaN,|,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2016-17,MIT (Mass. Inst. of Tech.),ACHA II,12.0,5.0,5.0,10.0,8.0,0.0,|,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2017-18,Did not play,NaN,NaN,NaN,NaN,NaN,NaN,NaN,|,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2018-19,MIT (Mass. Inst. of Tech.),ACHA III,8.0,5.0,10.0,15.0,8.0,NaN,|,NaN,NaN,NaN,NaN,NaN,NaN,NaN


_solução do vídeo_

In [7]:
import pandas as pd

table = webpage.select("table.hockey-stats")[0] # Busca a tabela com a classe `hockey-stats`
columns = table.find("thead").find_all("th") # Aqui monta as colunas do `DataFrame` com base no cabeçalho da tabela
column_names = [c.string for c in columns] # Busca os textos dos elementos

table_rows = table.find("tbody").find_all("tr") # Agora busca as linhas das tabelas para poder acessar os dados.
l = [] 
for tr in table_rows:
    td = tr.find_all('td') # Percorrendo as linhas é possível acessar os dados das tabelas com `td`.
    row = [str(tr.get_text()).strip() for tr in td] # Em objetos que possuem mais de um elemento o `get_text` é 
                                                    # mais adequado do que o `string` para não retornar `None`.
                                                    # `strip`: Remove espaços em branco da `str`.
    l.append(row) # Adiciona os dados na lista que será usada como valores do `DataFrame`

df = pd.DataFrame(l, columns=column_names) # Monta o `DataFrame` com as colunas e valores acessados.
df.head()

,S,Team,League,GP,G,A,TP,PIM,+/-,,POST,GP,G,A,TP,PIM,+/-
0,2014-15,MIT (Mass. Inst. of Tech.),ACHA II,17,3,9,12,20,,|,,,,,,,
1,2015-16,MIT (Mass. Inst. of Tech.),ACHA II,9,1,1,2,2,,|,,,,,,,
2,2016-17,MIT (Mass. Inst. of Tech.),ACHA II,12,5,5,10,8,0,|,,,,,,,
3,2017-18,Did not play,,,,,,,,|,,,,,,,
4,2018-19,MIT (Mass. Inst. of Tech.),ACHA III,8,5,10,15,8,,|,,,,,,,


### 3 - Pegue todos os _fun facts_ que usam a palavra _is_.

_minha solução_

In [8]:
import re

fun_facts = webpage.select('ul.fun-facts')[0] # Busca o elemento que possui os _fun facts_.
fun_facts_items = fun_facts.find_all('li') # Acessa a lista de items `li`.

regex = re.compile('is') # Define a regEx para `is`.

is_fun_facts = list(filter(lambda li: regex.search(li.get_text()), fun_facts_items)) # Filtra conforme aqueles
                                                                                        # atendem a regEx utilizando
                                                                                        # o método `search` do pacote
                                                                                        # _built in_ `re`.
is_fun_facts

[<li>Middle name is Ronald</li>,
 <li>Dunkin Donuts coffee is better than Starbucks</li>,
 <li>A favorite book series of mine is <i>Ender's Game</i></li>,
 <li>Current video game of choice is <i>Rocket League</i></li>,
 <li>The band that I've seen the most times live is the <i>Zac Brown Band</i></li>]

_solução do vídeo_

In [9]:
import re

facts = webpage.select("ul.fun-facts li") # Já acessa diretamente os elementos `li` para não ter que fazer o `find_all`
                                            # adiciona igual foi feito na minha solução.

facts_with_is = [fact.find(string=re.compile("is")) for fact in facts] # Retorna alguns itens `None` mas retorna os 
                                                                        # elementos que contem `is`.]

facts_with_is = [fact.find_parent().get_text() for fact in facts_with_is if fact] # Novamente list comprehension
                                                                                    # Utiliza o `if` depois do `in`
                                                                                    # para filtrar a lista antes de
                                                                                    # percorrê-la
                                                                                    # 
                                                                                    # Em seguida ao percorrer a lista
                                                                                    # utiliza o método `find_parent`
                                                                                    # para buscar o elemento que está 
                                                                                    # acima, assim se pode usar o 
                                                                                    # método `get_text` do _parent_
                                                                                    # para buscar todo o texto 
                                                                                    # (conteúdo) do elemento.

facts_with_is

['Middle name is Ronald',
 'Dunkin Donuts coffee is better than Starbucks',
 "A favorite book series of mine is Ender's Game",
 'Current video game of choice is Rocket League',
 "The band that I've seen the most times live is the Zac Brown Band"]

### 4 - Utilizar Beautiful Soup para ajudar a baixar uma imagem em uma página web.


_minha solução_

In [10]:
img = webpage.find('img', attrs={'alt': 'Lake Como'}) # Buscamos a imagem definida pelo atributo `alt`.
src = img['src'] # Acessa o caminho da imagem.

file_extension = src.split('.')[1] # Pegamos a extensão do arquivo que é definida por `.`

image = requests.get(f'https://keithgalli.github.io/web-scraping/{src}') # Acessa a imagem de maneira semelhante
                                                                            # de como acessamos o HTML, no fim
                                                                            # são tudo arquivos (ou bytes).
                                                                            # Como é uma pagina HTML e a imagem
                                                                            # referencia a partir da raiz do HTML
                                                                            # a url será a mesma que a do HTML
                                                                            # mas ao invés de referenciar a página
                                                                            # referenciei o caminho da imagem que é
                                                                            # como o servidor acessa a imagem.

with open(f'Lake como.{file_extension}', "wb") as file: # Cria um arquivo com a função open onde o modo de escrita é
                                            # em _byte_ (`"wb"`).
                                            # O nome do caminho do arquivo é definido no primerio parâmetro junto com
                                            # a extensão.
    file.write(image.content) # `image.content` retorna o binário da imagem acessada.

_solução do vídeo_

In [11]:
import requests # pip install requests
from bs4 import BeautifulSoup as bs # pip install beautifulsoup4

# Carrega a página web
url = 'https://keithgalli.github.io/web-scraping/'
r = requests.get(url+"webpage.html")

# Converte para um objeto beautiful soup
webpage = bs(r.content)

images = webpage.select("div.row div.column img")
image_url = images[0]['src']
full_url = url + image_url

img_data = requests.get(full_url).content
with open('lake_como.jpg', 'wb') as handler:
    handler.write(img_data)

### 5 - Resolva o desafio misterioso!

_minha solução_

In [12]:
challenge_links = webpage.select('[href^="challenge"]') # Selecionando os links que começam com `challenge`.

url = 'https://keithgalli.github.io/web-scraping/'

for link in challenge_links:
    file_link = url + link['href']
    link_page_content = requests.get(file_link).content

    link_page = bs(link_page_content)

    print(link_page.select_one('p#secret-word').string) # O select_one, seleciona um elemento, como há apenas um em
                                                        # cada página ou o que precisamos é o primeiro, logo funciona.

Make
sure
to
smash
that
like
button
and
subscribe
!!!


_solução do video_

In [13]:
files = webpage.select("div.block a")
relative_files = [f['href'] for f in files] # Usou list comprehension para acessar os links, como eu iria iterar 
                                            # de qualquer forma deixei tudo em um só laço `for`

url = 'https://keithgalli.github.io/web-scraping/'

for f in relative_files:
    full_url = url + f
    page = requests.get(full_url)
    bs_page = bs(page.content)
    secret_word_element = bs_page.find("p", attrs={"id": "secret-word"}) # Utiliza o `find` com `attrs` para buscar pelo
                                                                            # id
    secret_word = secret_word_element.string # Acessa o `string` por ter utilizado o `find` que retorna um elemento
                                                # e esse elemento possui apenas um conteúdo, permitindo usar `string`.
    print(secret_word)

Make
sure
to
smash
that
like
button
and
subscribe
!!!
